In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [3]:
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train.csv")
print(f"Shape ban đầu: {train.shape}")

Đang đọc dữ liệu train.csv...
Shape ban đầu: (699960, 27)


In [5]:
# Khởi tạo cột fold với giá trị mặc định
train["fold"] = -1

# Thiết lập 5-Fold StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

target_col = "is_fraud"

# Gán nhãn fold từ 0 đến 4 cho từng mẫu
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train, train[target_col])):
    train.loc[val_idx, "fold"] = fold_idx

print("Phân bố số lượng mẫu theo từng fold:")
print(train["fold"].value_counts().sort_index())

print("\nTỉ lệ nhãn gian lận (is_fraud) trên từng fold:")
print(train.groupby("fold")[target_col].mean())

Phân bố số lượng mẫu theo từng fold:
fold
0    139992
1    139992
2    139992
3    139992
4    139992
Name: count, dtype: int64

Tỉ lệ nhãn gian lận (is_fraud) trên từng fold:
fold
0    0.016694
1    0.016701
2    0.016701
3    0.016701
4    0.016701
Name: is_fraud, dtype: float64


In [6]:
# Lưu lại file train.csv đã có thêm cột fold
train.to_csv("train.csv", index=False)
print(f"Đã cập nhật và lưu train.csv thành công! Shape mới: {train.shape}")

Đã cập nhật và lưu train.csv thành công! Shape mới: (699960, 27)


In [4]:
# Cài đặt isotree (thư viện C++ đa luồng cực nhanh cho Extended Isolation Forest):
# !pip install isotree

from isotree import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Chuẩn bị dữ liệu bỏ nhãn is_fraud, id và fold
feature_cols = [c for c in train.columns if c not in ["id", "is_fraud", "fold"]]
X_train_unsupervised = train[feature_cols].copy()

# 2. Xử lý dữ liệu phân loại (Categorical Data):
# Chuyển các cột text và boolean sang kiểu 'category' để isotree tự động nhận diện và xử lý chia nhánh tối ưu
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
bool_cols = X_train_unsupervised.select_dtypes(include=["bool"]).columns.tolist()

for col in cat_cols + bool_cols:
    if col in X_train_unsupervised.columns:
        X_train_unsupervised[col] = X_train_unsupervised[col].astype("category")

# 3. Khởi tạo mô hình Extended Isolation Forest với cấu hình Scale Data & Categ Split chuyên sâu:
# - ndim=3: Kích hoạt siêu phẳng nghiêng ngẫu nhiên kết hợp 3 chiều đặc trưng (Extended IF)
# - standardize_data=True: Scale/chuẩn hóa dữ liệu số tại mỗi node để các đặc trưng có thang đo chênh lệch (như balance và hours) không áp đảo split
# - coefs="normal": Sinh trọng số hệ số nghiêng theo phân phối chuẩn (Gaussian) cân bằng hơn
# - categ_split_type="subset": Chia tập các danh mục thành 2 tập con ngẫu nhiên (tối ưu nhất cho Extended IF khi ndim > 1)
# - all_perm=False: Tăng tốc chia nhánh biến phân loại mà vẫn đảm bảo độ ngẫu nhiên cao
# - missing_action="auto": Tự động gán nhánh cho các giá trị khuyết thiếu (NaN)
# - n_jobs=-1: Tận dụng toàn bộ các nhân CPU xử lý 700.000 dòng
print("Bắt đầu huấn luyện Extended Isolation Forest...")
eif_model = IsolationForest(
    ndim=3,                  # ndim > 1: Extended Isolation Forest (kết hợp ngẫu nhiên 3 chiều)
    ntrees=200,              # Số lượng cây cách ly
    sample_size=256,         # Kích thước lấy mẫu cho mỗi cây
    standardize_data=True,   # TỰ ĐỘNG SCALE / CHUẨN HÓA DỮ LIỆU SỐ trước khi tạo siêu phẳng
    coefs="normal",          # Trọng số siêu phẳng ngẫu nhiên theo phân phối Gaussian
    categ_split_type="subset",# XỬ LÝ DỮ LIỆU PHÂN LOẠI: chia thành các tập con đa chiều
    missing_action="auto",   # Tự động xử lý missing values
    n_jobs=-1,               # Chạy đa luồng song song trên toàn bộ CPU
    random_seed=42
)

# Huấn luyện hoàn toàn KHÔNG GIÁM SÁT (không truyền nhãn)
eif_model.fit(X_train_unsupervised)
print("Huấn luyện xong Extended Isolation Forest!")

# 4. Dự đoán điểm bất thường (Anomaly Score, điểm càng cao càng bất thường/fraud)
train["anomaly_score_eif"] = eif_model.predict(X_train_unsupervised)

# 5. Đánh giá khả năng phát hiện gian lận tự thân không giám sát so với nhãn thực tế
y_true = train["is_fraud"].astype(int)
roc_auc = roc_auc_score(y_true, train["anomaly_score_eif"])
auc_pr = average_precision_score(y_true, train["anomaly_score_eif"])

print(f"ROC-AUC của Extended Isolation Forest (Unsupervised): {roc_auc:.4f}")
print(f"AUC-PR  của Extended Isolation Forest (Unsupervised): {auc_pr:.4f}")

Bắt đầu huấn luyện Extended Isolation Forest...
Huấn luyện xong Extended Isolation Forest!
ROC-AUC của Extended Isolation Forest (Unsupervised): 0.8036
AUC-PR  của Extended Isolation Forest (Unsupervised): 0.0818


In [5]:
train.to_csv("train_with_anomaly_score.csv", index=False)
print("Đã lưu train_with_anomaly_score.csv với cột anomaly_score_eif!")

Đã lưu train_with_anomaly_score.csv với cột anomaly_score_eif!
